# Módulo 03 · Lista de Exercícios

> **Manual de Estudos Interativo** · Trilha Engenharia de Software & Dados
> Projeto transversal: **Atlas / Aurora Comércio**

## Como usar esta lista

SQL se aprende escrevendo consultas que dão errado e descobrindo por quê.

**Duas regras:**

1. **Escreva antes de rodar.** Formule a consulta inteira mentalmente, depois execute. Se você vai por tentativa e erro, não está aprendendo o modelo — está adivinhando.
2. **Confira o resultado.** Uma consulta que roda não é uma consulta correta. Um `JOIN` mal feito devolve números lindos e errados. Pergunte sempre: *esse número faz sentido?*

| Nível | Aula base | Exercícios |
|-------|-----------|------------|
| 1 | 03_01 — Modelagem e DDL | 1–8 |
| 2 | 03_02 — Consultas básicas | 9–20 |
| 3 | 03_03 — Joins e subconsultas | 21–32 |
| 4 | 03_04 — Manutenção e transações | 33–40 |
| ⭐ | Perguntas de negócio | 41–50 |
| 🏆 | Projeto | Atlas sobre SQLite |

> ▶️ **Execute a célula de preparação abaixo antes de começar.**

In [ ]:
import sqlite3
import random

# ═══════════════════════════════════════════════════════════════
#  Banco de treino da Aurora Comércio
#  Execute esta célula UMA VEZ, antes de qualquer outra.
#  Ela é idempotente: pode rodar de novo a qualquer momento.
# ═══════════════════════════════════════════════════════════════

CONN = sqlite3.connect(":memory:")
CONN.execute("PRAGMA foreign_keys = ON")

CONN.executescript("""
CREATE TABLE categorias (
    id           INTEGER PRIMARY KEY,
    nome         TEXT    NOT NULL UNIQUE,
    margem_alvo  REAL    NOT NULL DEFAULT 0.25 CHECK (margem_alvo BETWEEN 0 AND 1)
);

CREATE TABLE produtos (
    id           INTEGER PRIMARY KEY,
    sku          TEXT    NOT NULL UNIQUE,
    nome         TEXT    NOT NULL,
    categoria_id INTEGER NOT NULL REFERENCES categorias(id) ON DELETE RESTRICT,
    preco        REAL    NOT NULL CHECK (preco >= 0),
    custo        REAL    NOT NULL CHECK (custo >= 0),
    estoque      INTEGER NOT NULL DEFAULT 0 CHECK (estoque >= 0),
    ativo        INTEGER NOT NULL DEFAULT 1 CHECK (ativo IN (0,1))
);

CREATE TABLE clientes (
    id            INTEGER PRIMARY KEY,
    nome          TEXT NOT NULL,
    email         TEXT NOT NULL UNIQUE,
    cidade        TEXT NOT NULL,
    uf            TEXT NOT NULL CHECK (length(uf) = 2),
    segmento      TEXT NOT NULL DEFAULT 'varejo'
                       CHECK (segmento IN ('varejo','corporativo')),
    data_cadastro TEXT NOT NULL,
    telefone      TEXT
);

CREATE TABLE pedidos (
    id          INTEGER PRIMARY KEY,
    cliente_id  INTEGER NOT NULL REFERENCES clientes(id) ON DELETE RESTRICT,
    data_pedido TEXT    NOT NULL,
    status      TEXT    NOT NULL CHECK (status IN ('pago','pendente','cancelado')),
    canal       TEXT    NOT NULL CHECK (canal IN ('site','app','marketplace')),
    frete       REAL    NOT NULL DEFAULT 0 CHECK (frete >= 0)
);

CREATE TABLE itens_pedido (
    id             INTEGER PRIMARY KEY,
    pedido_id      INTEGER NOT NULL REFERENCES pedidos(id)  ON DELETE CASCADE,
    produto_id     INTEGER NOT NULL REFERENCES produtos(id) ON DELETE RESTRICT,
    quantidade     INTEGER NOT NULL CHECK (quantidade > 0),
    preco_unitario REAL    NOT NULL CHECK (preco_unitario >= 0),
    UNIQUE (pedido_id, produto_id)
);
""")

_CATEGORIAS = [(1, "Notebooks", 0.18), (2, "Monitores", 0.22), (3, "Periféricos", 0.38),
               (4, "Armazenamento", 0.30), (5, "Redes", 0.28), (6, "Áudio", 0.35)]

_PRODUTOS = [
    ("NB-DELL-15",  "Notebook Dell Inspiron 15",     1, 2599.90, 2120.00,  14),
    ("NB-ACER-N5",  "Notebook Acer Nitro 5",         1, 3299.00, 2780.00,   7),
    ("NB-LEN-IP3",  "Notebook Lenovo IdeaPad 3",     1, 2199.00, 1850.00,  22),
    ("NB-APPL-M2",  "MacBook Air M2",                1, 9499.00, 8300.00,   3),
    ("MO-LG-24UW",  "Monitor LG 24 UltraWide",       2, 1199.00,  920.00,  31),
    ("MO-SAM-ODY",  "Monitor Samsung Odyssey 27",    2, 1849.00, 1420.00,  12),
    ("MO-AOC-22",   "Monitor AOC 22 Full HD",        2,  749.00,  560.00,  45),
    ("PE-LOG-MX3",  "Mouse Logitech MX Master 3",    3,  549.00,  340.00,  88),
    ("PE-LOG-M170", "Mouse Logitech M170",           3,   89.90,   52.00, 240),
    ("PE-RED-K552", "Teclado Redragon K552",         3,  249.00,  150.00,  64),
    ("PE-LOG-C920", "Webcam Logitech C920",          3,  449.00,  290.00,  37),
    ("AU-HYP-CL2",  "Headset HyperX Cloud II",       6,  399.00,  255.00,  29),
    ("AU-JBL-T510", "Fone JBL Tune 510BT",           6,  229.00,  140.00,  73),
    ("AR-SSD-1TB",  "SSD NVMe 1TB Kingston",         4,  489.00,  360.00,  52),
    ("AR-SSD-480",  "SSD SATA 480GB Sandisk",        4,  229.00,  158.00,  96),
    ("AR-HD-2TB",   "HD Externo 2TB Seagate",        4,  549.00,  410.00,  18),
    ("AR-PEN-128",  "Pendrive 128GB Sandisk",        4,   79.90,   44.00, 180),
    ("RE-TPL-AX55", "Roteador TP-Link Archer AX55",  5,  699.00,  505.00,  26),
    ("RE-TPL-RE30", "Repetidor TP-Link RE305",       5,  229.00,  152.00,  41),
    ("RE-INT-AX20", "Placa de Rede Intel AX200",     5,  189.00,  124.00,  33),
]

_NOMES = ["Ana Costa", "Bruno Rocha", "Carla Dias", "Daniel Souza", "Elisa Martins",
          "Fábio Nunes", "Gustavo Reis", "Helena Prado", "Igor Batista", "Julia Andrade",
          "Lucas Moreira", "Maria Souza", "Nathalia Freitas", "Otávio Pinto",
          "Priscila Gomes", "Rafael Torres", "Sabrina Melo", "Thiago Barros",
          "Vanessa Lima", "William Cruz", "Beatriz Almeida", "Caio Ferreira",
          "Débora Ramos", "Eduardo Pires", "Fernanda Vieira", "Gabriel Mendes",
          "Isabela Rocha", "João Lima", "Karina Duarte", "Leonardo Castro"]

_CIDADES = [("Campinas", "SP"), ("São Paulo", "SP"), ("Sorocaba", "SP"),
            ("Ribeirão Preto", "SP"), ("Jundiaí", "SP"), ("Santos", "SP"),
            ("Belo Horizonte", "MG"), ("Uberlândia", "MG"), ("Curitiba", "PR"),
            ("Londrina", "PR"), ("Porto Alegre", "RS"), ("Florianópolis", "SC"),
            ("Rio de Janeiro", "RJ"), ("Niterói", "RJ"), ("Salvador", "BA"),
            ("Recife", "PE"), ("Fortaleza", "CE"), ("Brasília", "DF"),
            ("Goiânia", "GO"), ("Vitória", "ES")]

_rnd = random.Random(42)     # semente fixa = todos veem os mesmos números

CONN.executemany("INSERT INTO categorias VALUES (?,?,?)", _CATEGORIAS)
CONN.executemany(
    "INSERT INTO produtos (sku,nome,categoria_id,preco,custo,estoque,ativo) "
    "VALUES (?,?,?,?,?,?,1)", _PRODUTOS)

_acentos = str.maketrans("áéíóúãõâêôç", "aeiouaoaeoc")
_clientes = []
for _i, _nome in enumerate(_NOMES, 1):
    _cidade, _uf = _rnd.choice(_CIDADES)
    _login = _nome.split()[0].lower().translate(_acentos)
    _clientes.append((
        _i, _nome, f"{_login}{_i}@email.com", _cidade, _uf,
        "corporativo" if _rnd.random() < 0.25 else "varejo",
        f"2026-{_rnd.randint(1, 6):02d}-{_rnd.randint(1, 28):02d}",
        f"(19) 9{_rnd.randint(1000, 9999)}-{_rnd.randint(1000, 9999)}" if _rnd.random() < 0.6 else None,
    ))
CONN.executemany("INSERT INTO clientes VALUES (?,?,?,?,?,?,?,?)", _clientes)

_pedidos, _itens, _id_item = [], [], 0
# Os 3 últimos clientes ficam SEM pedido de propósito: toda base real tem
# gente que se cadastrou e nunca comprou, e você precisa saber encontrá-los.
for _pid in range(1, 181):
    _mes = _rnd.choices([5, 6, 7], weights=[2, 3, 4])[0]
    _pedidos.append((
        _pid, _rnd.randint(1, len(_NOMES) - 3),
        f"2026-{_mes:02d}-{_rnd.randint(1, 28):02d}",
        _rnd.choices(["pago", "pendente", "cancelado"], weights=[80, 12, 8])[0],
        _rnd.choices(["site", "app", "marketplace"], weights=[50, 30, 20])[0],
        _rnd.choice([0.0, 9.90, 19.90, 29.90]),
    ))
    _n_itens = _rnd.choices([1, 2, 3, 4], weights=[45, 30, 17, 8])[0]
    for _prod in _rnd.sample(range(1, len(_PRODUTOS) + 1), _n_itens):
        _id_item += 1
        _itens.append((
            _id_item, _pid, _prod,
            _rnd.choices([1, 2, 3, 5, 10], weights=[55, 22, 12, 7, 4])[0],
            round(_PRODUTOS[_prod - 1][3] * _rnd.choice([1.0, 1.0, 1.0, 0.95, 0.90]), 2),
        ))
CONN.executemany("INSERT INTO pedidos VALUES (?,?,?,?,?,?)", _pedidos)
CONN.executemany("INSERT INTO itens_pedido VALUES (?,?,?,?,?)", _itens)
CONN.commit()


# ── Funções auxiliares ───────────────────────────────────────
def _fmt(valor):
    if valor is None:
        return "NULL"
    if isinstance(valor, float):
        return f"{valor:,.2f}"
    if isinstance(valor, int):
        return f"{valor:,}"
    return str(valor)


def sql(consulta, parametros=(), limite=30):
    """Executa uma consulta e imprime o resultado formatado."""
    try:
        cursor = CONN.execute(consulta, parametros)
    except sqlite3.Error as erro:
        print(f"❌ {type(erro).__name__}: {erro}")
        return None

    if cursor.description is None:
        CONN.commit()
        print(f"✅ OK — {cursor.rowcount} linha(s) afetada(s)" if cursor.rowcount >= 0 else "✅ OK")
        return None

    colunas = [d[0] for d in cursor.description]
    linhas = cursor.fetchall()
    total = len(linhas)
    linhas = linhas[:limite]
    if not linhas:
        print("(nenhuma linha)")
        return []

    texto = [[_fmt(v) for v in linha] for linha in linhas]
    numerica = [
        any(isinstance(l[i], (int, float)) for l in linhas)
        and all(isinstance(l[i], (int, float)) or l[i] is None for l in linhas)
        for i in range(len(colunas))
    ]
    larguras = [max(len(colunas[i]), max(len(l[i]) for l in texto))
                for i in range(len(colunas))]

    def borda(e, m, d):
        return e + m.join("─" * (w + 2) for w in larguras) + d

    print(borda("┌", "┬", "┐"))
    print("│ " + " │ ".join(c.ljust(w) for c, w in zip(colunas, larguras)) + " │")
    print(borda("├", "┼", "┤"))
    for linha in texto:
        print("│ " + " │ ".join(
            (v.rjust(w) if numerica[i] else v.ljust(w))
            for i, (v, w) in enumerate(zip(linha, larguras))) + " │")
    print(borda("└", "┴", "┘"))
    print(f"{total} linha(s)" + (f" — exibindo as {limite} primeiras" if total > limite else ""))
    return linhas


def ddl(script):
    """Executa um script com vários comandos."""
    try:
        CONN.executescript(script)
        CONN.commit()
        print("✅ Script executado")
    except sqlite3.Error as erro:
        print(f"❌ {type(erro).__name__}: {erro}")


print("✅ Banco da Aurora criado em memória\n")
sql("""
SELECT 'categorias'   AS tabela, COUNT(*) AS linhas FROM categorias
UNION ALL SELECT 'produtos',     COUNT(*) FROM produtos
UNION ALL SELECT 'clientes',     COUNT(*) FROM clientes
UNION ALL SELECT 'pedidos',      COUNT(*) FROM pedidos
UNION ALL SELECT 'itens_pedido', COUNT(*) FROM itens_pedido
""")

---
# NÍVEL 1 — Modelagem e DDL
*Aula `03_01`*

### 1. Modelando do zero

A Aurora quer controlar **devoluções**. Regras do negócio:

- Uma devolução refere-se a **um item específico de um pedido**
- Tem data, quantidade devolvida e motivo
- O motivo vem de uma lista fechada: `defeito`, `arrependimento`, `divergência`, `avaria no transporte`
- A quantidade devolvida não pode ser maior que a comprada (essa regra o banco não consegue garantir sozinho — explique por quê)
- Devolução pode ter status: `solicitada`, `aprovada`, `recusada`, `concluída`

Escreva o `CREATE TABLE` completo, com PK, FK, `CHECK` e `DEFAULT` apropriados.

In [ ]:
# 1.

### 2. Chave natural vs artificial

A tabela `clientes` usa `id INTEGER PRIMARY KEY` e tem `email` como `UNIQUE`.

a) Escreva uma versão alternativa usando `email` como chave primária.
b) Insira dois clientes e crie uma tabela `pedidos` referenciando essa PK.
c) Simule a troca de e-mail de um cliente. O que acontece com os pedidos?
d) Em uma célula markdown, explique qual das duas modelagens você defenderia e por quê.

In [ ]:
# 2.

<!-- 2.d — sua resposta aqui -->

### 3. Constraints defendendo o negócio

Adicione à tabela `produtos` (recriando-a) as seguintes garantias:

a) O preço nunca pode ser menor que o custo
b) A margem, quando calculada, não pode passar de 90%
c) O SKU deve ter exatamente 10 ou 11 caracteres e começar com duas letras maiúsculas seguidas de hífen
d) O estoque não pode passar de 10.000 unidades

Teste **cada uma** tentando violá-la.

In [ ]:
# 3.

### 4. Ações referenciais

Para cada relação abaixo, escolha entre `RESTRICT`, `CASCADE` e `SET NULL`, e **justifique**:

| Relação | Ação | Justificativa |
|---------|------|---------------|
| `itens_pedido.pedido_id` → `pedidos.id` | | |
| `pedidos.cliente_id` → `clientes.id` | | |
| `produtos.categoria_id` → `categorias.id` | | |
| `produtos.fornecedor_id` → `fornecedores.id` (opcional) | | |
| `devolucoes.item_pedido_id` → `itens_pedido.id` | | |

<!-- 4. — preencha a tabela aqui -->

### 5. Afinidade de tipo

a) Crie uma tabela normal e grave texto numa coluna `INTEGER`. Confirme com `typeof()`.
b) Crie a mesma tabela com `STRICT` e prove que agora dá erro.
c) Crie uma terceira versão sem `STRICT`, mas com `CHECK (typeof(coluna) = 'integer')`. Funciona?
d) Qual das três abordagens você usaria em produção?

In [ ]:
# 5.

### 6. NULL na prática

Preveja o resultado de cada expressão **antes** de executar. Depois execute e explique as que você errou.

```sql
SELECT
    NULL = NULL,
    NULL <> NULL,
    NULL IS NULL,
    5 + NULL,
    'a' || NULL,
    COALESCE(NULL, NULL, 'terceiro'),
    NULLIF(10, 10),
    NULLIF(10, 5),
    CASE WHEN NULL THEN 'sim' ELSE 'nao' END,
    COUNT(NULL),
    MAX(NULL)
```

In [ ]:
# 6.

### 7. Inspecionando o schema

Escreva consultas que respondam, usando `sqlite_master` e `PRAGMA`:

a) Quantas tabelas existem no banco?
b) Quais colunas de `pedidos` são obrigatórias (`NOT NULL`)?
c) Quais são todas as chaves estrangeiras de `itens_pedido`, e para onde apontam?
d) Quais índices existem, e em quais colunas?
e) Qual o DDL original da tabela `produtos`?

In [ ]:
# 7.

### 8. Diagrama ER

Desenhe (em texto/ASCII, numa célula markdown) o diagrama ER do modelo da Aurora **incluindo a tabela de devoluções** que você criou no exercício 1.

Marque as chaves primárias, estrangeiras e as cardinalidades.

<!-- 8. — seu diagrama aqui -->

---
# NÍVEL 2 — Consultas básicas
*Aula `03_02`*

### 9. Filtros combinados

Liste os produtos que satisfaçam **todas** as condições: categoria 1, 2 ou 4; preço entre R$ 500 e R$ 3.000; estoque acima de 10; ativos.

Mostre `sku`, `nome`, `preco`, `estoque` e a margem em percentual, ordenados pela maior margem.

In [ ]:
# 9.

### 10. Busca textual robusta

Encontre todos os produtos cujo nome contenha "ssd" **ou** "hd", ignorando maiúsculas e minúsculas de forma confiável (lembre-se do problema com acentos).

In [ ]:
# 10.

### 11. A ordem de execução

Explique, em markdown, por que cada consulta abaixo funciona ou falha:

```sql
-- (a)
SELECT preco * 2 AS dobro FROM produtos WHERE dobro > 1000;

-- (b)
SELECT preco * 2 AS dobro FROM produtos ORDER BY dobro DESC;

-- (c)
SELECT uf, COUNT(*) AS n FROM clientes WHERE n > 2 GROUP BY uf;

-- (d)
SELECT uf, COUNT(*) AS n FROM clientes GROUP BY uf HAVING n > 2;
```

Depois execute cada uma para confirmar sua previsão.

In [ ]:
# 11.

<!-- 11. — suas explicações aqui -->

### 12. Agregações e NULL

a) Quantos clientes existem no total?
b) Quantos têm telefone cadastrado?
c) Qual a taxa de preenchimento do telefone, em percentual com 1 casa?
d) Se você calculasse `AVG(length(telefone))`, o denominador seria o total de clientes ou só os que têm telefone? Prove.

In [ ]:
# 12.

### 13. `GROUP BY` com múltiplas dimensões

Faturamento por **UF e canal**, apenas de pedidos pagos. Mostre UF, canal, nº de pedidos, nº de itens, faturamento e ticket médio. Ordene por UF e faturamento decrescente.

In [ ]:
# 13.

### 14. `WHERE` vs `HAVING`

Escreva **uma** consulta que use os dois corretamente: liste as cidades com mais de 2 pedidos **pagos** e faturamento acima de R$ 20.000.

Em um comentário, explique qual condição foi para o `WHERE`, qual foi para o `HAVING` e por quê.

In [ ]:
# 14.

### 15. Pivô com `CASE`

Monte uma tabela com uma linha por **categoria** e colunas para o faturamento em cada **canal** (`site`, `app`, `marketplace`), mais o total.

In [ ]:
# 15.

### 16. Classificação em faixas

Classifique os **clientes** por valor total comprado:

| Faixa | Critério |
|-------|----------|
| Diamante | ≥ R$ 30.000 |
| Ouro | R$ 15.000 a R$ 29.999 |
| Prata | R$ 5.000 a R$ 14.999 |
| Bronze | < R$ 5.000 |

Mostre a contagem e o faturamento total de cada faixa.

In [ ]:
# 16.

### 17. Séries temporais

a) Faturamento por mês.
b) Faturamento por dia da semana (traduza o número para o nome).
c) Ticket médio por mês.
d) Qual dia da semana vende mais?

In [ ]:
# 17.

### 18. Divisão inteira

a) Calcule `COUNT(*) / 2` para os clientes. O resultado está certo?
b) Corrija de três formas diferentes.
c) Calcule o percentual de clientes corporativos — primeiro do jeito errado, depois do certo.

In [ ]:
# 18.

### 19. Paginação

a) Liste os produtos ordenados por preço decrescente, 5 por página. Mostre as páginas 1, 2 e 3.
b) Em um comentário, explique por que `OFFSET` fica lento em tabelas grandes.
c) Reescreva a página 2 usando paginação por cursor (`WHERE preco < ultimo_preco`). Que problema surge se houver preços empatados?

In [ ]:
# 19.

### 20. Formatação de saída

Produza um relatório em que cada linha seja **uma única string** formatada assim:

```
Campinas/SP ......... R$ 82.145,30 (12 pedidos)
```

Use `||`, `printf` e funções de texto. *(Dica: `printf('%-20s', x)` alinha à esquerda.)*

In [ ]:
# 20.

---
# NÍVEL 3 — Joins e subconsultas
*Aula `03_03`*

### 21. Join básico com verificação

Liste os 20 itens de pedido de maior valor, mostrando: pedido, data, cliente, cidade, produto, categoria, quantidade, preço unitário e total.

**Depois confira:** a soma dos totais dessa consulta bate com `SUM(quantidade * preco_unitario)` da tabela `itens_pedido`?

In [ ]:
# 21.

### 22. A inflação do JOIN

a) Calcule `SUM(frete)` diretamente na tabela `pedidos`.
b) Calcule `SUM(p.frete)` depois de juntar com `itens_pedido`.
c) Explique a diferença.
d) Escreva uma consulta que traga, **corretamente**, o faturamento **e** o frete total por canal.

In [ ]:
# 22.

### 23. `LEFT JOIN` e zeros

Liste **todas** as categorias com: nº de produtos, nº de produtos vendidos, unidades vendidas e receita. Categorias sem venda devem aparecer com zero, não sumir.

In [ ]:
# 23.

### 24. `WHERE` vs `ON`

a) Escreva uma consulta com `LEFT JOIN` e uma condição da tabela da direita no `WHERE`. Conte as linhas.
b) Mova a condição para o `ON`. Conte de novo.
c) Explique a diferença numericamente.

In [ ]:
# 24.

### 25. Anti-joins

Responda cada uma com anti-join **ou** `NOT EXISTS`:

a) Quais produtos nunca foram vendidos?
b) Quais clientes nunca fizeram pedido?
c) Quais clientes fizeram pedido mas nenhum foi pago?
d) Quais categorias não têm nenhum produto com estoque?
e) Quais produtos nunca foram vendidos pelo canal `marketplace`?

In [ ]:
# 25.

### 26. `NOT IN` vs `NOT EXISTS`

Demonstre o problema do `NOT IN` com `NULL`:

a) Escreva uma consulta com `NOT IN` sobre uma subconsulta **sem** NULL. Funciona.
b) Faça a subconsulta devolver um `NULL`. O que acontece?
c) Reescreva com `NOT EXISTS`.
d) Reescreva com anti-join.
e) Meça o tempo das três versões com `time.perf_counter()`.

In [ ]:
# 26.

### 27. Self join

a) Pares de produtos da mesma categoria cuja diferença de preço seja menor que R$ 60.
b) Clientes da mesma cidade e do mesmo segmento.
c) Pares de pedidos do mesmo cliente feitos com menos de 7 dias de intervalo.

In [ ]:
# 27.

### 28. Subconsulta correlacionada → JOIN

Reescreva esta consulta usando `LEFT JOIN` + `GROUP BY`, e compare os tempos:

```sql
SELECT
    pr.nome,
    (SELECT COUNT(*)       FROM itens_pedido i WHERE i.produto_id = pr.id) AS vezes,
    (SELECT SUM(quantidade) FROM itens_pedido i WHERE i.produto_id = pr.id) AS unidades,
    (SELECT MAX(preco_unitario) FROM itens_pedido i WHERE i.produto_id = pr.id) AS maior_preco
FROM produtos pr
ORDER BY unidades DESC;
```

In [ ]:
# 28.

### 29. CTEs encadeadas

Usando **pelo menos 3 CTEs**, monte um relatório com:

- Uma linha por cliente
- Nome, cidade, segmento
- Nº de pedidos, faturamento, ticket médio
- A posição do cliente no ranking de faturamento
- O percentual que ele representa do total

In [ ]:
# 29.

### 30. Variação mês a mês

Usando CTEs, calcule o faturamento por mês e a variação percentual em relação ao mês anterior.

*(Dica: faça um self join da CTE mensal consigo mesma, casando `mes` com o mês anterior. Como você calcula "o mês anterior" em formato `AAAA-MM`?)*

In [ ]:
# 30.

### 31. CTE recursiva

a) Gere a série de números de 1 a 20.
b) Gere todos os dias de junho/2026 e mostre o faturamento diário, com zeros nos dias sem venda.
c) Gere uma grade completa mês × categoria (3 meses × 7 categorias = 21 linhas), mesmo onde não houve venda.

In [ ]:
# 31.

### 32. Operações de conjunto

a) `UNION ALL`: uma lista única com os 5 produtos mais caros e os 5 mais baratos, com uma coluna indicando qual grupo.
b) `INTERSECT`: clientes que compraram em Notebooks **e** em Áudio.
c) `EXCEPT`: cidades que tiveram pedidos em junho mas **nenhum** em julho.

In [ ]:
# 32.

---
# NÍVEL 4 — Manutenção e transações
*Aula `03_04`*

### 33. SQL injection

a) Escreva uma função Python `buscar_cliente_inseguro(email)` que concatene a string.
b) Demonstre três ataques diferentes: vazar todos os registros, sempre retornar verdadeiro, e um `UNION` que vaze outra tabela.
c) Reescreva como `buscar_cliente_seguro(email)` com placeholder.
d) Prove que os três ataques falham na versão segura.

In [ ]:
# 33.

### 34. `UPDATE` com protocolo

Aplique um reajuste de 7% em todos os produtos com margem abaixo de 20%.

**Siga o protocolo:**

1. `SELECT` com o mesmo `WHERE`, contando as linhas
2. `SELECT` mostrando o valor antes e o valor depois (sem alterar nada)
3. Só então o `UPDATE`, dentro de uma transação
4. Confira o resultado
5. Faça `ROLLBACK` e confirme que voltou ao original

In [ ]:
# 34.

### 35. Soft delete completo

a) Adicione `deletado_em` a `clientes`.
b) Marque como removidos os clientes sem nenhum pedido.
c) Crie uma view `clientes_ativos`.
d) Escreva uma consulta que use a view.
e) Escreva uma consulta de auditoria que mostre os removidos.
f) "Restaure" um deles.

In [ ]:
# 35.

### 36. `UPSERT` idempotente

a) Crie `metricas_diarias(dia, pedidos, faturamento)` com PK em `dia`.
b) Popule com `INSERT ... SELECT ... ON CONFLICT DO UPDATE`.
c) Rode **três vezes** e prove que a contagem de linhas não muda.
d) Insira um pedido novo e reprocesse. A métrica do dia foi atualizada?

In [ ]:
# 36.

### 37. Índices e planos

a) Rode `EXPLAIN QUERY PLAN` numa consulta que filtre `clientes.segmento`. O que aparece?
b) Crie o índice adequado e rode de novo.
c) Crie um índice composto `(segmento, uf)` e teste três consultas: só por segmento, por segmento e UF, e só por UF. Qual não usa o índice? Por quê?
d) Crie um índice parcial só para clientes corporativos e compare.

In [ ]:
# 37.

### 38. Quando o índice não ajuda

Demonstre com `EXPLAIN QUERY PLAN` três casos em que um índice existente **não** é usado, e corrija cada um:

a) Função aplicada à coluna
b) `LIKE '%texto'`
c) Expressão aritmética sobre a coluna (`WHERE preco * 1.1 > 500`)

In [ ]:
# 38.

### 39. Transação atômica

Escreva `transferir_estoque(origem_id, destino_id, quantidade)` que:

- Debita do produto origem e credita no destino
- Registra a movimentação numa tabela `movimentacoes_estoque`
- Faz tudo em **uma** transação

Teste três cenários: sucesso, estoque insuficiente e produto inexistente. Em cada falha, prove que **nada** foi alterado.

In [ ]:
# 39.

### 40. `SAVEPOINT`

Escreva uma carga em lote que processe 5 registros, sendo o 3º inválido:

- Cada registro tem seu `SAVEPOINT`
- Se um falhar, faz `ROLLBACK TO` daquele savepoint e continua com os próximos
- Ao final, `COMMIT` do que deu certo
- Reporte quantos entraram e quantos falharam

In [ ]:
# 40.

---
# ⭐ NÍVEL INTEGRADO — Perguntas de negócio

Cada exercício é uma pergunta que alguém da Aurora faria. Escreva **uma** consulta que responda.

### 41. Curva ABC de produtos

Classifique os produtos por receita acumulada: A (até 80%), B (até 95%), C (o resto). Mostre produto, receita, share, share acumulado e classe.

In [ ]:
# 41.

### 42. Cesta de compras

Quais pares de produtos são comprados juntos com mais frequência? Mostre os 10 primeiros com o nome dos dois produtos e quantas vezes apareceram no mesmo pedido.

In [ ]:
# 42.

### 43. Análise de recência

Para cada cliente: última compra, dias desde então, nº de pedidos e valor total. Classifique em `Ativo` (≤30 dias), `Esfriando` (31–60), `Em risco` (61–90) e `Perdido` (>90), tomando 2026-08-01 como "hoje".

In [ ]:
# 43.

### 44. Produtos canibalizando

Encontre produtos da mesma categoria com preços a menos de 15% de distância um do outro, onde **um vende muito mais** que o outro. São candidatos a descontinuar.

In [ ]:
# 44.

### 45. Contribuição de margem

Ranking de produtos por **margem absoluta gerada** (não por receita). Mostre também a margem percentual e compare a posição no ranking de receita com a posição no ranking de margem.

In [ ]:
# 45.

### 46. Cobertura de estoque

Para cada produto: estoque atual, média de vendas diárias nos últimos 90 dias, cobertura em dias e um alerta (`repor já`, `atenção`, `ok`, `excesso`). Ordene pelos mais críticos.

In [ ]:
# 46.

### 47. Penetração por praça

Para cada cidade: quantos clientes cadastrados, quantos compraram, taxa de conversão, e o faturamento médio por cliente ativo.

In [ ]:
# 47.

### 48. Concentração de receita

Que percentual do faturamento vem dos 20% maiores clientes? (Princípio de Pareto.) Mostre também os 20% menores.

In [ ]:
# 48.

### 49. Coorte de aquisição

Agrupe os clientes pelo **mês de cadastro** e mostre, para cada coorte: quantos clientes, quantos compraram, ticket médio e faturamento total.

In [ ]:
# 49.

### 50. Relatório executivo completo

Escreva **uma única consulta** (com CTEs) que produza um resumo de uma linha por mês, contendo:

- Mês
- Pedidos totais, pagos, cancelados
- Taxa de cancelamento
- Faturamento
- Ticket médio
- Clientes únicos
- Produtos distintos vendidos
- Variação percentual do faturamento vs. mês anterior
- Cidade campeã do mês

In [ ]:
# 50.

---
---

# 🏆 PROJETO DO MÓDULO — Atlas sobre SQLite

## Contexto

> *"Os dados estão em 14 planilhas diferentes. A do comercial tem 'Campinas', a do financeiro tem 'campinas/SP', a do estoque tem 'CPS'. Quando alguém corrige o preço de um produto, corrige em uma planilha só. Ontem descobrimos que o mesmo cliente aparece 4 vezes com e-mails diferentes."*
> — Diretora de Operações

## Objetivo

Migrar o Atlas de arquivos CSV para um **banco relacional SQLite**, com schema modelado, carga idempotente e relatórios em SQL.

O programa deve continuar funcionando pela CLI — mas agora lendo do banco.

## Parte A — Modelagem

1. Desenhe o diagrama ER em `docs/MODELAGEM.md`, incluindo cardinalidades e justificando cada decisão.
2. Escreva `dados/schema.sql` com o DDL completo:
   - `categorias`, `produtos`, `clientes`, `pedidos`, `itens_pedido`
   - PKs artificiais, FKs com ação referencial deliberada
   - `CHECK` em todo domínio fechado (status, canal, UF, quantidades, preços)
   - `NOT NULL` e `UNIQUE` onde couber
   - Comentários explicando as decisões não óbvias
3. Escreva `dados/indices.sql` com os índices, **e um comentário justificando cada um**.

**Pergunta a responder no documento:** por que `itens_pedido` guarda `preco_unitario` se `produtos` já tem `preco`?

## Parte B — Camada de acesso

Crie `src/atlas/repositorio.py`:

```python
@contextmanager
def conectar(caminho): ...        # PRAGMA foreign_keys, WAL, row_factory

@contextmanager
def transacao(conexao): ...       # commit/rollback

def criar_schema(conexao): ...
def upsert_cliente(conexao, ...) -> int: ...
def upsert_produto(conexao, ...) -> int: ...
def inserir_pedido(conexao, ...) -> int: ...
def inserir_itens(conexao, pedido_id, itens): ...
```

**Requisitos inegociáveis:**

- 🔴 **Toda** consulta parametrizada. Zero concatenação.
- Todo SQL vive nesta camada. Nenhuma string SQL em outro arquivo.
- Toda escrita dentro de transação.

## Parte C — Migração

Crie `src/atlas/migracao.py` que leia os CSVs do M01 e popule o banco:

1. Normaliza texto (cidade, nome, e-mail, status)
2. Deduplica clientes por e-mail e produtos por nome
3. Usa `UPSERT` — rodar duas vezes não duplica nada
4. Rejeita linhas inválidas sem derromper o processo, gravando o motivo
5. Tudo em uma transação por arquivo
6. Relata: linhas lidas, inseridas, atualizadas, rejeitadas

**Critério de aceitação:** rodar a migração 3 vezes seguidas produz exatamente o mesmo estado do banco.

## Parte D — Relatórios em SQL

Crie `dados/consultas/` com um arquivo `.sql` por relatório:

| Arquivo | Relatório |
|---------|-----------|
| `faturamento_por_cidade.sql` | Ranking com share |
| `faturamento_por_categoria.sql` | Com margem |
| `top_produtos.sql` | Por receita e por margem |
| `top_clientes.sql` | Com recência |
| `evolucao_mensal.sql` | Com variação |
| `curva_abc.sql` | Praças em A/B/C |
| `alerta_estoque.sql` | Cobertura em dias |
| `qualidade_dados.sql` | Órfãos, duplicatas, inconsistências |

E `src/atlas/relatorios_sql.py`, que lê o `.sql`, executa e formata a saída.

> 💡 **Por que SQL em arquivo separado?** Porque assim o analista de negócio consegue ler e ajustar a consulta sem mexer em Python — e você consegue testar a consulta direto no cliente do banco.

## Parte E — CLI atualizada

```bash
python main.py migrar dados/brutos/vendas_jul2026.csv   # CSV -> banco
python main.py relatorio faturamento_por_cidade         # roda uma consulta
python main.py relatorio --todos                        # roda todas
python main.py schema --recriar                         # recria o banco do zero
```

## Parte F — Comparação

Em `docs/CSV_VS_SQL.md`, compare as duas implementações:

| Aspecto | CSV (M01) | SQLite (M03) |
|---------|-----------|--------------|
| Linhas de código do relatório por cidade | | |
| Tempo de execução | | |
| O que acontece com dado inválido | | |
| Como adicionar uma nova dimensão de análise | | |
| Como dois processos leem ao mesmo tempo | | |
| Onde mora a regra "faturamento só conta pagos" | | |

Meça de verdade. Números, não impressões.

## Critérios de avaliação

| Critério | Peso |
|----------|------|
| Schema bem modelado (PK, FK, CHECK, ações referenciais) | 25% |
| Migração idempotente e resiliente a dado sujo | 25% |
| 100% das consultas parametrizadas | 15% |
| Consultas SQL corretas (números batem com o M01) | 15% |
| Índices justificados | 10% |
| Documentação (modelagem + comparação) | 10% |

> 🔴 **Reprovação automática:** qualquer SQL montado por concatenação de string com dado externo.

## Desafios extras

- ⭐ Uma view `vw_vendas` que já traga tudo junto, usada pelos relatórios
- ⭐ Triggers que atualizem o estoque automaticamente ao inserir um item
- ⭐ Tabela `auditoria` alimentada por trigger em `UPDATE`/`DELETE` de produtos
- ⭐⭐ Sistema de migração de schema versionado (`001_inicial.sql`, `002_add_devolucoes.sql`) com controle de versão aplicada
- ⭐⭐ Comparação de desempenho: mesma consulta com e sem índice, medida com `time.perf_counter()`

## Entrega

Ao terminar, este comando deve funcionar em uma máquina limpa:

```bash
git clone <seu-repo> && cd atlas
./scripts/setup.sh
python main.py schema --recriar
python main.py migrar dados/brutos/vendas_jul2026.csv
python main.py relatorio --todos
```

E os números devem **bater** com os do relatório do Módulo 01.

In [ ]:
# 🏆 Espaço de trabalho do projeto.
# Sugestão: desenvolva em projeto_Atlas/ pelo VS Code e use esta célula
# só para prototipar consultas antes de colocá-las nos arquivos .sql

---

## ✅ Autoavaliação do Módulo 03

**Modelagem**

- [ ] Identifico as entidades de um domínio e desenho o ER
- [ ] Escolho PK artificial e protejo chaves naturais com `UNIQUE`
- [ ] Uso `CHECK` para todo domínio fechado
- [ ] Escolho a ação referencial certa em cada FK
- [ ] Sei por que uma tabela de junção pode ter atributos próprios

**Consultas**

- [ ] Recito a ordem lógica de execução
- [ ] Domino `GROUP BY` + `HAVING`
- [ ] Uso `CASE WHEN`, inclusive para pivotar
- [ ] Sei que o `JOIN` multiplica linhas e me protejo
- [ ] Escolho `LEFT JOIN` quando os zeros importam
- [ ] Escrevo CTEs em vez de subconsultas aninhadas
- [ ] Uso `NOT EXISTS` em vez de `NOT IN`

**Escrita**

- [ ] 🔴 **100% das minhas consultas são parametrizadas**
- [ ] Escrevo o `SELECT` antes de todo `UPDATE`/`DELETE`
- [ ] Uso `UPSERT` para cargas idempotentes
- [ ] Envolvo escritas relacionadas em transação com rollback

**Desempenho**

- [ ] Sei quando criar um índice e quando não criar
- [ ] Leio `EXPLAIN QUERY PLAN`
- [ ] Reconheço as situações em que o índice não é usado

**Autonomia**

- [ ] Pego uma pergunta de negócio e escrevo a consulta sem tentativa e erro
- [ ] Desconfio de números que parecem bons demais e sei conferi-los

---

### ➡️ Próximo módulo

**Módulo 04 — Python Avançado.** Dor da Aurora: *"O script virou um monstro de 800 linhas."* Você vai refatorar o Atlas para orientação a objetos, com tipagem e logging estruturado.